# BI-LSTM Model
## NER Model

In [1]:
# import the libraries
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt')  # for tokenizing sentences

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, TimeDistributed, Dense, Dropout # TIME DISTRIBUTED IS USED FOR TAG EACH WORD

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [2]:
# Step 2: Load Dataset
df = pd.read_csv("ner_1000_dataset.csv")  # CSV should have columns: Sentence, Entity, Tag
df

,Sentence,Entity,Tag
0,John Smith visited New York last summer.,John Smith,PERSON
1,John Smith visited New York last summer.,New York,LOCATION
2,John Smith visited Paris last summer.,John Smith,PERSON
3,John Smith visited Paris last summer.,Paris,LOCATION
4,John Smith visited London last summer.,John Smith,PERSON
...,...,...,...
995,"The conference by Google is on January 10, 202...","January 10, 2024",DATE
996,"The conference by Google is on March 5, 2023 o...","March 5, 2023",DATE
997,"The conference by Google is on July 14, 2022 o...","July 14, 2022",DATE
998,"The conference by Google is on December 1, 202...","December 1, 2021",DATE


In [3]:
# Step 3: Convert Sentences to Token-Level BIO Tags
X_tokens = []  # tokenized sentences
y_labels = []  # BIO tags

In [4]:
y_labels

[]

In [5]:
import nltk
nltk.download('punkt_tab')

# Step 3: Convert sentences to tokens & BIO tags (simpler version)
X_tokens = []
y_labels = []

for sentence, group in df.groupby('Sentence'):
    tokens = nltk.word_tokenize(sentence)        # split sentence into words
    bio_labels = ['O'] * len(tokens)            # start with all 'O'

    # Go through each entity in this sentence
    for _, row in group.iterrows():
        entity_words = row['Entity'].split()    # split entity into words
        tag = row['Tag']

        # Try to find where the entity starts in the sentence
        for i in range(len(tokens)):
            # Check if the next words match the entity
            if tokens[i:i+len(entity_words)] == entity_words:
                bio_labels[i] = 'B-' + tag      # first word = B-Tag
                for j in range(1, len(entity_words)):
                    bio_labels[i+j] = 'I-' + tag  # rest = I-Tag

    X_tokens.append(tokens)
    y_labels.append(bio_labels)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [6]:
X_tokens

[['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '184',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '299',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '414',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '529',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '644',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '69',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '759',
  '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'Berlin',
  'on',
  'day',
  '874',
  '.'],
 ['Amazon', 'opened', 'a', 'new', 'office', 'in', 'Berlin', '.'],
 ['Amazon',
  'opened',
  'a',
  'new',
  'office',
  'in',
  'London',
  'on',
  'day'

In [7]:
# Step 4: Encode Tokens & Labels
# Convert words to numbers
tokenizer = Tokenizer(lower=False, oov_token='OOV')
tokenizer.fit_on_texts(X_tokens)
X_seq = tokenizer.texts_to_sequences(X_tokens) # converts words -> numbers
max_len = max(len(s) for s in X_seq)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')

In [8]:
# Convert BIO labels to numbers
all_labels = list({label for seq in y_labels for label in seq})
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

LabelEncoder()

In [9]:
y_seq = [label_encoder.transform(seq) for seq in y_labels] # converts each sentences BIO tag into a list of integer-encoded labels
y_pad = pad_sequences(y_seq, maxlen=max_len, padding='post', value=label_encoder.transform(['O'])[0])
y_cat = np.array([to_categorical(seq, num_classes=len(all_labels)) for seq in y_pad])

In [10]:
print("Vocabulary size:", len(tokenizer.word_index)+1)
print("Number of unique tags:", len(all_labels))
print("Max sequence length:", max_len)

Vocabulary size: 963
Number of unique tags: 11
Max sequence length: 14


In [11]:
# Step 5: Build BiLSTM Model
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=70, input_length=max_len))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.3))
model.add(TimeDistributed(Dense(len(all_labels), activation='softmax'))) # TIME DISTRIBUTION WILL GIVE TAG FOR EACH WORD

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
# Step 6: Train the Model
history = model.fit(
    np.array(X_pad),
    y_cat,
    batch_size=32,
    epochs=10,       # increase epochs
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 86ms/step - accuracy: 0.8742 - loss: 1.1987 - val_accuracy: 0.9542 - val_loss: 0.3280
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.9110 - loss: 0.3775 - val_accuracy: 0.9542 - val_loss: 0.3324
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.9110 - loss: 0.3279 - val_accuracy: 0.9542 - val_loss: 0.3684
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9110 - loss: 0.2985 - val_accuracy: 0.9542 - val_loss: 0.4390
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.9107 - loss: 0.2689 - val_accuracy: 0.9542 - val_loss: 0.4703
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.9131 - loss: 0.2304 - val_accuracy: 0.9707 - val_loss: 0.4647
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - accuracy: 0.9215 - loss: 0.2009 - val_accuracy: 0.9707 - val_loss: 0.4573
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - accuracy: 0.9277 - loss: 0.1716 - val_accuracy: 0.9707 - 

In [13]:
# Step 7: Predict NER Tags for New Sentences
def predict_ner(sentence):
    tokens = nltk.word_tokenize(sentence)
    seq = tokenizer.texts_to_sequences([tokens])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(padded, verbose=0)
    pred_labels = [label_encoder.inverse_transform([np.argmax(p)])[0] for p in pred[0]]
    for word, tag in zip(tokens, pred_labels[:len(tokens)]):
        print(word, tag)

# Example Predictions
print("\nPrediction Example 1:")
predict_ner("Paris is the capital of France.")



Prediction Example 1:
Paris B-PERSON
is O
the O
capital O
of O
France B-LOCATION
. O
